In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import copy

In [2]:
DATA_DIR = Path(r"C:/Users/57713/Desktop/AI/coursework/data/EBM-NLP/2/ebm_nlp_2_00")

In [3]:
def get_doc_ids(split="train", label_type="participants"):
    if split == "test":
        split = "test/gold"

    ann_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type
        / split
    )

    doc_ids = [p.stem.split(".")[0] for p in ann_dir.glob("*.AGGREGATED.ann")]
    return sorted(doc_ids)


def load_labels_for_doc(doc_id, label_type="participants", split="train"):
    if split == "test":
        split = "test/gold"

    ann_path = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type
        / split
        / f"{doc_id}.AGGREGATED.ann"
    )

    if not ann_path.exists():
        return None

    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f]

    return labels


def load_labels(doc_ids, label_type="participants", split="train"):
    labels = []
    for doc_id in doc_ids:
        cur = load_labels_for_doc(doc_id, label_type, split)
        if cur is not None:
            labels.append(cur)
    return labels


def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return f.read().strip().split()


def load_documents(doc_ids):
    return [load_document(doc_id) for doc_id in doc_ids]

In [4]:
def hierarchical_to_bio(tags):
    bio = []
    prev = 0

    for t in tags:
        t = int(t)
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B")
            else:
                bio.append("I")
        prev = t

    return bio


def convert_all_labels_to_bio(labels):
    return [hierarchical_to_bio(doc_labels) for doc_labels in labels]


def merge_joint_labels(labels_p, labels_i, labels_o):
    joint_labels = []
    overlap_count = 0

    for doc_p, doc_i, doc_o in zip(labels_p, labels_i, labels_o):
        cur_doc = []

        for lp, li, lo in zip(doc_p, doc_i, doc_o):
            active = []
            if lp != 'O':
                active.append(('P', lp))
            if li != 'O':
                active.append(('I', li))
            if lo != 'O':
                active.append(('O', lo))

            if len(active) == 0:
                cur_doc.append('O')
            elif len(active) == 1:
                field, bio = active[0]
                cur_doc.append(f"{bio}-{field}")
            else:
                priority = {'P': 0, 'I': 1, 'O': 2}
                active = sorted(active, key=lambda x: priority[x[0]])
                field, bio = active[0]
                cur_doc.append(f"{bio}-{field}")
                overlap_count += 1

        joint_labels.append(cur_doc)

    return joint_labels, overlap_count

In [5]:
def build_vocab(all_tokens, min_freq=1):
    freq = {}
    for doc in all_tokens:
        for tok in doc:
            tok = tok.lower()
            freq[tok] = freq.get(tok, 0) + 1

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for tok, count in freq.items():
        if count >= min_freq:
            vocab[tok] = len(vocab)
    return vocab


def encode_tokens(tokens, vocab):
    return [[vocab.get(tok.lower(), vocab["<UNK>"]) for tok in doc] for doc in tokens]


def encode_labels(labels, label2id):
    return [[label2id[x] for x in doc_labels] for doc_labels in labels]


def pad_sequences(seqs, pad_value=0, max_len=None):
    if max_len is None:
        max_len = max(len(x) for x in seqs)

    padded = []
    masks = []

    for seq in seqs:
        seq = seq[:max_len]
        pad_len = max_len - len(seq)
        padded.append(seq + [pad_value] * pad_len)
        masks.append([1] * len(seq) + [0] * pad_len)

    return padded, masks


def decode_predictions(pred_ids, id2label, masks):
    all_preds = []

    for doc_preds, doc_mask in zip(pred_ids, masks):
        cur_labels = []
        for p, m in zip(doc_preds, doc_mask):
            if m == 1:
                cur_labels.append(id2label[int(p)])
        all_preds.append(cur_labels)

    return all_preds


def apply_mask_to_labels(labels, masks):
    new_labels = []
    for doc_labels, doc_mask in zip(labels, masks):
        cur = []
        for lab, m in zip(doc_labels, doc_mask):
            if m == 1:
                cur.append(lab)
        new_labels.append(cur)
    return new_labels


def flatten_labels(labels):
    flat = []
    for doc in labels:
        flat.extend(doc)
    return flat

In [6]:
def build_token_dataloader_joint(max_len=400, dev_size=0.15, random_state=42, batch_size=16):
    train_doc_ids = None
    test_doc_ids = None

    for label_type in ['participants', 'interventions', 'outcomes']:
        cur_train_ids = set(get_doc_ids(split="train", label_type=label_type))
        cur_test_ids = set(get_doc_ids(split="test", label_type=label_type))

        if train_doc_ids is None:
            train_doc_ids = cur_train_ids
            test_doc_ids = cur_test_ids
        else:
            train_doc_ids = train_doc_ids & cur_train_ids
            test_doc_ids = test_doc_ids & cur_test_ids

    train_doc_ids = sorted(list(train_doc_ids))
    test_doc_ids = sorted(list(test_doc_ids))

    train_doc_ids, dev_doc_ids = train_test_split(
        train_doc_ids,
        test_size=dev_size,
        random_state=random_state,
        shuffle=True
    )

    train_doc_ids = sorted(train_doc_ids)
    dev_doc_ids = sorted(dev_doc_ids)

    print("Official train document count:", len(train_doc_ids) + len(dev_doc_ids))
    print("Train split document count:", len(train_doc_ids))
    print("Dev split document count:", len(dev_doc_ids))
    print("Official test document count:", len(test_doc_ids))

    train_labels_p = convert_all_labels_to_bio(load_labels(train_doc_ids, 'participants', 'train'))
    dev_labels_p = convert_all_labels_to_bio(load_labels(dev_doc_ids, 'participants', 'train'))
    test_labels_p = convert_all_labels_to_bio(load_labels(test_doc_ids, 'participants', 'test'))

    train_labels_i = convert_all_labels_to_bio(load_labels(train_doc_ids, 'interventions', 'train'))
    dev_labels_i = convert_all_labels_to_bio(load_labels(dev_doc_ids, 'interventions', 'train'))
    test_labels_i = convert_all_labels_to_bio(load_labels(test_doc_ids, 'interventions', 'test'))

    train_labels_o = convert_all_labels_to_bio(load_labels(train_doc_ids, 'outcomes', 'train'))
    dev_labels_o = convert_all_labels_to_bio(load_labels(dev_doc_ids, 'outcomes', 'train'))
    test_labels_o = convert_all_labels_to_bio(load_labels(test_doc_ids, 'outcomes', 'test'))

    train_tokens = load_documents(train_doc_ids)
    dev_tokens = load_documents(dev_doc_ids)
    test_tokens = load_documents(test_doc_ids)

    train_joint_labels, overlap_train = merge_joint_labels(train_labels_p, train_labels_i, train_labels_o)
    dev_joint_labels, overlap_dev = merge_joint_labels(dev_labels_p, dev_labels_i, dev_labels_o)
    test_joint_labels, overlap_test = merge_joint_labels(test_labels_p, test_labels_i, test_labels_o)

    print("train overlap:", overlap_train)
    print("dev overlap:", overlap_dev)
    print("test overlap:", overlap_test)

    label2id = {
        'O': 0,
        'B-P': 1, 'I-P': 2,
        'B-I': 3, 'I-I': 4,
        'B-O': 5, 'I-O': 6
    }

    id2label = {v: k for k, v in label2id.items()}

    vocab = build_vocab(train_tokens)

    X_train = encode_tokens(train_tokens, vocab)
    X_dev = encode_tokens(dev_tokens, vocab)
    X_test = encode_tokens(test_tokens, vocab)

    y_train = encode_labels(train_joint_labels, label2id)
    y_dev = encode_labels(dev_joint_labels, label2id)
    y_test = encode_labels(test_joint_labels, label2id)

    X_train_pad, X_train_mask = pad_sequences(X_train, pad_value=vocab['<PAD>'], max_len=max_len)
    X_dev_pad, X_dev_mask = pad_sequences(X_dev, pad_value=vocab['<PAD>'], max_len=max_len)
    X_test_pad, X_test_mask = pad_sequences(X_test, pad_value=vocab['<PAD>'], max_len=max_len)

    y_train_pad, _ = pad_sequences(y_train, pad_value=-100, max_len=max_len)
    y_dev_pad, _ = pad_sequences(y_dev, pad_value=-100, max_len=max_len)
    y_test_pad, _ = pad_sequences(y_test, pad_value=-100, max_len=max_len)

    X_train_pad = torch.tensor(X_train_pad, dtype=torch.long)
    X_dev_pad = torch.tensor(X_dev_pad, dtype=torch.long)
    X_test_pad = torch.tensor(X_test_pad, dtype=torch.long)
    y_train_pad = torch.tensor(y_train_pad, dtype=torch.long)
    y_dev_pad = torch.tensor(y_dev_pad, dtype=torch.long)
    y_test_pad = torch.tensor(y_test_pad, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(X_train_pad, y_train_pad), batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(TensorDataset(X_dev_pad, y_dev_pad), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(X_test_pad, y_test_pad), batch_size=batch_size)

    return (
    vocab,
    id2label,
    train_loader,
    dev_loader,
    test_loader,
    X_dev_mask,
    X_test_mask,
    dev_joint_labels,
    test_joint_labels,
    dev_doc_ids,
    test_doc_ids,
    dev_tokens,
    test_tokens
)


In [7]:
class BiLSTMTagger(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        hidden_dim,
        num_labels,
        embedding_matrix=None,
        pad_idx=0,
        dropout=0.3  
    ):
        super().__init__()

        # embedding
        if embedding_matrix is not None:
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(embedding_matrix, dtype=torch.float),
                freeze=False,
                padding_idx=pad_idx
            )
        else:
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        self.dropout = nn.Dropout(dropout)

        # LSTM
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x = self.dropout(x)
        x, _ = self.lstm(x)
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits

In [8]:
from sklearn.metrics import f1_score

def compute_macro_f1(model, data_loader, ignore_index=-100):
    device = next(model.parameters()).device
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for input_ids, labels in data_loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            logits = model(input_ids)
            preds = torch.argmax(logits, dim=-1)

            preds = preds.view(-1)
            labels = labels.view(-1)

            valid_mask = labels != ignore_index

            all_preds.extend(preds[valid_mask].cpu().numpy())
            all_labels.extend(labels[valid_mask].cpu().numpy())

    return f1_score(all_labels, all_preds, average="macro")

In [9]:
def compute_class_weights_from_loader(train_loader, num_classes, ignore_index=-100, mode="sqrt_inv"):
    counts = torch.zeros(num_classes, dtype=torch.float)

    for _, labels in train_loader:
        valid = labels[labels != ignore_index].view(-1)
        for c in range(num_classes):
            counts[c] += (valid == c).sum().item()

    counts = torch.clamp(counts, min=1.0)

    if mode == "inv":
        weights = counts.sum() / counts
    elif mode == "sqrt_inv":
        weights = torch.sqrt(counts.sum() / counts)
    elif mode == "log_inv":
        weights = torch.log1p(counts.sum() / counts)
    else:
        raise ValueError("mode must be 'inv', 'sqrt_inv', or 'log_inv'")

    weights = weights / weights.mean()
    return weights


def train_model(model, train_loader, dev_loader=None, epochs=5, lr=1e-3, weight_mode="sqrt_inv"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    num_classes = model.classifier.out_features
    weights = compute_class_weights_from_loader(
        train_loader,
        num_classes=num_classes,
        ignore_index=-100,
        mode=weight_mode
    ).to(device)

    print("Class weights:", weights)

    criterion = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)

    best_state = None
    best_dev_f1 = -1.0

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0

        for input_ids, labels in train_loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            logits = model(input_ids)

            loss = criterion(
                logits.view(-1, logits.shape[-1]),
                labels.view(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        if dev_loader is not None:
            model.eval()
            total_dev_loss = 0

            with torch.no_grad():
                for input_ids, labels in dev_loader:
                    input_ids = input_ids.to(device)
                    labels = labels.to(device)

                    logits = model(input_ids)
                    loss = criterion(
                        logits.view(-1, logits.shape[-1]),
                        labels.view(-1)
                    )

                    total_dev_loss += loss.item()

            avg_dev_loss = total_dev_loss / len(dev_loader)
            dev_macro_f1 = compute_macro_f1(model, dev_loader, ignore_index=-100)
            print(
                f"Epoch {epoch+1}, "
                f"Train Loss: {avg_train_loss:.4f}, "
                f"Dev Loss: {avg_dev_loss:.4f}, "
                f"Dev Macro F1: {dev_macro_f1:.4f}"
                )

            if dev_macro_f1 > best_dev_f1:
                best_dev_f1 = dev_macro_f1
                best_state = copy.deepcopy(model.state_dict())
        else:
            print(f"Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Loaded best model based on dev macro F1: {best_dev_f1:.4f}")

    return model


def predict(model, data_loader):
    device = next(model.parameters()).device
    model.eval()
    all_preds = []

    with torch.no_grad():
        for input_ids, _ in data_loader:
            input_ids = input_ids.to(device)

            logits = model(input_ids)
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())

    return all_preds

In [10]:
(
    vocab,
    id2label,
    train_loader,
    dev_loader,
    test_loader,
    X_dev_mask,
    X_test_mask,
    dev_labels,
    test_labels,
    dev_doc_ids,
    test_doc_ids,
    dev_tokens,
    test_tokens
) = build_token_dataloader_joint(max_len=400, dev_size=0.15, random_state=42)

print("vocab size:", len(vocab))

Official train document count: 4457
Train split document count: 3788
Dev split document count: 669
Official test document count: 184
train overlap: 5616
dev overlap: 831
test overlap: 59
vocab size: 39602


In [11]:
model = BiLSTMTagger(
    vocab_size=len(vocab),
    emb_dim=100,
    hidden_dim=256,
    num_labels=7,
    embedding_matrix=None
)

In [12]:
model = train_model(model, train_loader, dev_loader=dev_loader, epochs=5, lr=1e-3)

Class weights: tensor([0.2122, 1.6541, 1.1048, 1.2060, 0.8894, 1.1943, 0.7392])
Epoch 1, Train Loss: 1.2899, Dev Loss: 1.0128, Dev Macro F1: 0.4523
Epoch 2, Train Loss: 0.9946, Dev Loss: 0.9025, Dev Macro F1: 0.5003
Epoch 3, Train Loss: 0.8945, Dev Loss: 0.8479, Dev Macro F1: 0.5241
Epoch 4, Train Loss: 0.8335, Dev Loss: 0.8141, Dev Macro F1: 0.5199
Epoch 5, Train Loss: 0.7864, Dev Loss: 0.7958, Dev Macro F1: 0.5334
Loaded best model based on dev macro F1: 0.5334


In [13]:
pred_ids = predict(model, test_loader)
pred_labels = decode_predictions(pred_ids, id2label, X_test_mask)

In [14]:
def extract_single_type(labels, target_type):
    label_map = {
        "participants": "P",
        "interventions": "I",
        "outcomes": "O"
    }

    short_type = label_map[target_type]
    new_labels = []

    for doc in labels:
        cur = []
        for lab in doc:
            if lab == "O":
                cur.append("O")
            elif lab.endswith(f"-{short_type}"):
                cur.append(lab.split("-")[0])   # B or I
            else:
                cur.append("O")
        new_labels.append(cur)

    return new_labels

In [22]:
label_type = "interventions"

test_labels_base = extract_single_type(test_labels, label_type)
pred_labels_base = extract_single_type(pred_labels, label_type)

In [23]:
output_data = {
    "label_type": label_type,
    "doc_ids": test_doc_ids,
    "tokens": test_tokens,
    "gold_labels": test_labels_base,
    "pred_labels": pred_labels_base
}

In [24]:
print(output_data["label_type"])
print(output_data["doc_ids"][0])
print(output_data["tokens"][0][:10])
print(output_data["gold_labels"][0][:10])
print(output_data["pred_labels"][0][:10])

interventions
10070173
['Comparison', 'of', 'budesonide', 'Turbuhaler', 'with', 'budesonide', 'aqua', 'in', 'the', 'treatment']
['O', 'O', 'B', 'I', 'O', 'B', 'I', 'O', 'O', 'O']
['O', 'O', 'B', 'I', 'I', 'B', 'I', 'O', 'O', 'O']
